# Calcium Imaging — Batch Analysis
Scans `OUTPUTS_DIR` for experiment folders (those containing `dff.npy`), runs the full
analysis on each, exports per-experiment plots, and produces cross-experiment summary plots.

Epoch shading is applied to all time-series plots using the fixed recording protocol.

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
BASE_DIR      = r"Z:\ephacoffice\DColameo\Ca_Anand_AllData"
METADATA_XLSX = r'C:\Users\DColameo\Documents\dev\pyprojects\mps_ca\Application_period.xlsx'

# Co-activity
CORR_THRESHOLD   = 0.4
DIST_THRESHOLD   = 60

# Spike & burst detection (RMS-based)
SPIKE_RMS_FACTOR   = 5.0
BURST_RMS_FACTOR   = 1.2
RMS_WINDOW_S       = 60.0
BURST_MIN_DURATION = 0.5

# Video colormap
DFF_VMIN        = None
DFF_VMAX        = None
DFF_SCALE       = 1.0
ROI_VIDEO_FPS   = 15
ROI_VIDEO_ALPHA = 0.75
ROI_VIDEO_CMAP  = 'inferno'
MAX_VIDEO_FRAMES = None
RENDER_VIDEO    = True    # False = skip mp4 rendering

# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
from pathlib import Path
import re, traceback
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
import matplotlib.animation as _anim
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from scipy.ndimage import uniform_filter1d, label as nd_label
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.stats import zscore
from sklearn.decomposition import PCA
from skimage import measure
from skimage.color import label2rgb
import subprocess

plt.rcParams['figure.dpi'] = 110
base_root = Path(BASE_DIR)

def _get_genotype(name):
    if 'KO' in name: return 'KO'
    if 'WT' in name: return 'WT'
    return 'unknown'

def _savefig(fig, path):
    fig.savefig(str(path), dpi=120, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.close(fig)

def _shade_epochs(axes_list, t_max, epochs=None, alpha=0.12, add_labels=True):
    """Shade epoch bands; add text labels above the first axis."""
    if not epochs: return
    for ep in epochs:
        t0 = float(ep['start_s'])
        t1 = min(float(ep['end_s']), float(t_max))
        if t0 >= float(t_max): break
        for ax in axes_list:
            ax.axvspan(t0, t1, color=ep['color'], alpha=alpha, lw=0, zorder=0)
    if add_labels:
        ax_top = axes_list[0]
        for ep in epochs:
            t0 = float(ep['start_s'])
            t1 = min(float(ep['end_s']), float(t_max))
            if t0 >= float(t_max): break
            ax_top.text((t0+t1)/2, 1.01, ep['label'],
                        transform=ax_top.get_xaxis_transform(),
                        ha='center', va='bottom', fontsize=6.5, color='#333', clip_on=False)

print('Imports OK')
# Epoch color palette — assigned per unique application label
_EPOCH_PALETTE = [
    '#AED6F1',  # light blue  (low glucose / starvation)
    '#A9DFBF',  # light green (high glucose)
    '#FAE6B0',  # yellow      (high glucose + drug)
    '#F9C3C0',  # light red
    '#D7BDE2',  # light purple
    '#FDEBD0',  # peach
    '#D5DBDB',  # gray
]

def _assign_epoch_colors(epochs):
    seen = {}
    result = []
    for ep in epochs:
        key = ep.get('application', ep.get('label', ''))
        if key not in seen:
            seen[key] = _EPOCH_PALETTE[len(seen) % len(_EPOCH_PALETTE)]
        result.append({**ep, 'color': seen[key]})
    return result


def parse_excel_metadata(xlsx_path):
    import openpyxl, re as _re
    def _infer_sf(fname):
        m = _re.match(r'^(\d+_\d+)_', fname)
        return m.group(1) if m else None
    wb = openpyxl.load_workbook(xlsx_path)
    result = []
    for sheet in wb.sheetnames:
        ws = wb[sheet]
        rows = list(ws.iter_rows(values_only=True))[1:]
        exp_dir = cur_app = cur_phase = None
        files = []
        for row in rows:
            main_folder, subfolder, filename, application, phase, time_val, _, frames, seg = row
            if main_folder: exp_dir = str(main_folder)
            if application: cur_app = str(application)
            if phase: cur_phase = str(phase)
            if filename:
                fname = str(filename)
                if not fname.endswith('.tif'): fname += '.tif'
                inferred = _infer_sf(str(filename))
                files.append({'subfolder': inferred if inferred else str(subfolder) if subfolder else None,
                              'filename': fname, 'application': cur_app, 'phase': cur_phase,
                              'use_for_seg': bool(seg)})
        if exp_dir: result.append({'sheet': sheet, 'exp_dir': exp_dir, 'files': files})
    return result


_EPOCH_PALETTE = ['#AED6F1','#A9DFBF','#FAE6B0','#F9C3C0','#D7BDE2','#FDEBD0','#D5DBDB']

def _assign_epoch_colors(epochs):
    seen = {}
    result = []
    for ep in epochs:
        key = ep.get('application', ep.get('label', ''))
        if key not in seen: seen[key] = _EPOCH_PALETTE[len(seen) % len(_EPOCH_PALETTE)]
        result.append({**ep, 'color': seen[key]})
    return result


def _epochs_from_meta(meta, source_fps=4, target_fps=1, default_frames=1200):
    ratio = source_fps // target_fps
    out, t, prev, start_t = [], 0.0, None, 0.0
    for fi in meta['files']:
        dur_s = (default_frames // ratio) / target_fps
        key   = (fi['application'], fi['phase'])
        if key != prev:
            if prev: out.append({'application': prev[0], 'phase': prev[1],
                                  'label': f'{prev[0]} ({prev[1]})',
                                  'start_s': start_t, 'end_s': t})
            prev = key; start_t = t
        t += dur_s
    if prev: out.append({'application': prev[0], 'phase': prev[1],
                          'label': f'{prev[0]} ({prev[1]})',
                          'start_s': start_t, 'end_s': t})
    return out


print('Helpers loaded')

## Run batch

In [ ]:
def analyse_experiment(exp_dir, xl_meta=None):
    exp_dir   = Path(exp_dir) / 'pipeline_output'
    plots_dir = exp_dir / 'analysis_plots'
    plots_dir.mkdir(exist_ok=True)

    def _log(m): print(f'  [{exp_dir.name}] {m}')
    def _save(fig, name): _savefig(fig, plots_dir / name)

    # ---- Load --------------------------------------------------------
    roi_mask  = np.load(exp_dir / 'roi_mask.npy')
    centroids = np.load(exp_dir / 'centroids.npy')
    dff       = np.load(exp_dir / 'dff.npy')
    time_axis = np.load(exp_dir / 'time_axis.npy')
    mean_img  = tifffile.imread(str(exp_dir / 'mean_image.tif'))
    n_cells, T = dff.shape
    rprops     = measure.regionprops(roi_mask)
    fps_est    = T / time_axis[-1] if time_axis[-1] > 0 else 1.0
    t_max      = float(time_axis[-1])

    # Derive epochs from Excel metadata (no file dependency)
    _epochs = _assign_epoch_colors(_epochs_from_meta(xl_meta)) if xl_meta else []
    _log(f'{n_cells} cells, {T} frames ({t_max:.0f} s) '
         f'| epochs: {[e["label"] for e in _epochs] if _epochs else "none"}')

    # ---- Overview --------------------------------------------------------
    mean_norm = (mean_img - mean_img.min()) / (mean_img.max() - mean_img.min() + 1e-9)
    overlay   = np.clip(label2rgb(roi_mask, image=mean_norm, bg_label=0, alpha=0.4), 0, 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    p1, p99 = np.percentile(mean_img, [1, 99])
    axes[0].imshow(mean_img, cmap='gray', vmin=p1, vmax=p99)
    axes[0].set_title('Mean image'); axes[0].axis('off')
    axes[1].imshow(overlay)
    axes[1].set_title(f'ROIs (n={n_cells})')
    for rp in rprops:
        y, x = rp.centroid
        axes[1].text(x, y, str(rp.label), color='white', fontsize=5, ha='center', va='center')
    axes[1].axis('off')
    plt.suptitle(exp_dir.name, fontsize=9); plt.tight_layout()
    _save(fig, '01_overview.png')

    # ---- Correlation matrix ----------------------------------------------
    corr = np.corrcoef(dff)
    dist_corr = 1 - corr
    np.fill_diagonal(dist_corr, 0)
    dist_corr = np.clip(dist_corr, 0, None)
    dist_corr = (dist_corr + dist_corr.T) / 2
    Z_link = linkage(squareform(dist_corr), method='ward')
    order  = dendrogram(Z_link, no_plot=True)['leaves']
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, mat, title in zip(axes,
        [corr, corr[np.ix_(order, order)]],
        ['Correlation (original)', 'Correlation (clustered)']):
        im = ax.imshow(mat, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
        ax.set_title(title); ax.set_xlabel('Cell #'); ax.set_ylabel('Cell #')
        plt.colorbar(im, ax=ax, label='Pearson r')
    plt.suptitle(exp_dir.name, fontsize=9); plt.tight_layout()
    _save(fig, '02_correlation.png')
    mean_corr = float(corr[np.triu_indices(n_cells, k=1)].mean())

    # ---- Proximity vs co-activity ----------------------------------------
    dist_mat = squareform(pdist(centroids))
    triu_idx = np.triu_indices(n_cells, k=1)
    pdist_v  = dist_mat[triu_idx]
    pcorr_v  = corr[triu_idx]
    nearby   = pdist_v < DIST_THRESHOLD
    coactive = pcorr_v > CORR_THRESHOLD
    both     = nearby & coactive
    colors   = np.where(both, 'crimson', np.where(nearby, 'orange',
               np.where(coactive, 'steelblue', 'lightgray')))
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(pdist_v, pcorr_v, c=colors, s=10, alpha=0.6, linewidths=0)
    ax.axhline(CORR_THRESHOLD, color='steelblue', lw=1, ls='--')
    ax.axvline(DIST_THRESHOLD, color='orange',    lw=1, ls='--')
    ax.set_xlabel('Distance (px)'); ax.set_ylabel('Pearson r')
    ax.set_title(f'{exp_dir.name} -- proximity vs co-activity')
    plt.tight_layout()
    _save(fig, '03_proximity.png')

    # ---- Spike & burst detection (rolling RMS) ---------------------------
    rms_win       = max(3, int(RMS_WINDOW_S * fps_est))
    cell_rms      = np.sqrt(uniform_filter1d(dff**2, size=rms_win, axis=1, mode='nearest'))
    active_binary = dff > (SPIKE_RMS_FACTOR * cell_rms)
    pop_activity  = active_binary.mean(axis=0)
    pop_rms       = np.sqrt(uniform_filter1d(pop_activity**2, size=rms_win, mode='nearest'))
    min_frames    = max(1, int(BURST_MIN_DURATION * fps_est))
    burst_labeled, n_raw = nd_label(pop_activity > (BURST_RMS_FACTOR * pop_rms))
    bursts = []
    for b in range(1, n_raw + 1):
        frames = np.where(burst_labeled == b)[0]
        if len(frames) < min_frames: continue
        bursts.append({
            'frame_start': int(frames[0]), 'frame_end': int(frames[-1]),
            't_start': float(time_axis[frames[0]]), 't_end': float(time_axis[frames[-1]]),
            'duration_s': float(time_axis[frames[-1]] - time_axis[frames[0]]),
            'peak_frac' : float(pop_activity[frames].max()),
            'n_recruits': int(active_binary[:, frames].any(axis=1).sum()),
        })
    _log(f'{len(bursts)} bursts detected')

    # Per-epoch burst counts
    epoch_burst_counts = {}
    for ep in _epochs:
        if ep['start_s'] >= t_max: continue
        cnt = sum(1 for b in bursts if ep['start_s'] <= b['t_start'] < min(ep['end_s'], t_max))
        epoch_burst_counts[ep['label']] = epoch_burst_counts.get(ep['label'], 0) + cnt

    # Raster + pop trace
    burst_thr = BURST_RMS_FACTOR * pop_rms
    fig, axes = plt.subplots(2, 1, figsize=(16, 8),
                              gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
    ax_r, ax_p = axes
    for i in range(n_cells):
        t_on = time_axis[active_binary[i]]
        ax_r.scatter(t_on, np.full(len(t_on), i), s=1.5, c='k', linewidths=0)
    for b in bursts:
        ax_r.axvspan(b['t_start'], b['t_end'], color='salmon', alpha=0.3, lw=0)
    ax_r.set_ylabel('Cell #'); ax_r.invert_yaxis(); ax_r.set_ylim(n_cells-0.5, -0.5)
    ax_r.set_title(f'Spike raster ({SPIKE_RMS_FACTOR}x RMS) -- {len(bursts)} bursts')
    for s in ['top','right']: ax_r.spines[s].set_visible(False)
    ax_p.fill_between(time_axis, pop_activity, alpha=0.5, color='steelblue', label='pop. activity')
    ax_p.plot(time_axis, burst_thr, color='crimson', lw=1, ls='--', label=f'{BURST_RMS_FACTOR}x RMS')
    for b in bursts:
        ax_p.axvspan(b['t_start'], b['t_end'], color='salmon', alpha=0.3, lw=0)
    ax_p.set_xlabel('Time (s)'); ax_p.set_ylabel('Frac. active'); ax_p.legend(fontsize=8)
    for s in ['top','right']: ax_p.spines[s].set_visible(False)
    _shade_epochs([ax_r, ax_p], t_max, _epochs)
    plt.suptitle(exp_dir.name, fontsize=9); plt.tight_layout()
    _save(fig, '04_bursts.png')

    # ---- PCA (time-resolved + cell-resolved) ----------------------------
    Z = zscore(dff, axis=1)
    n_c = min(10, n_cells-1, T-1)
    n_t = min(10, n_cells-1, T-1)
    emb_t = PCA(n_components=n_t).fit_transform(Z.T)
    pca_c = PCA(n_components=n_c)
    emb_c = pca_c.fit_transform(Z)
    ev_c  = pca_c.explained_variance_ratio_ * 100
    burst_t = np.zeros(T, dtype=bool)
    for b in bursts:
        burst_t[b['frame_start']:b['frame_end']+1] = True
    _bp = np.zeros(n_cells)
    if bursts:
        for b in bursts:
            _bp += active_binary[:, b['frame_start']:b['frame_end']+1].mean(axis=1)

    # Assign frame epoch index for colouring
    frame_epoch = np.full(T, -1, dtype=int)
    ep_colors_list = []
    for ei, ep in enumerate(_epochs):
        mask = (time_axis >= ep['start_s']) & (time_axis < min(ep['end_s'], t_max + 1))
        frame_epoch[mask] = ei
        if ei not in [e for e in range(ei) if _epochs[e]['label'] == ep['label']]:
            ep_colors_list.append((ei, ep['label'], ep['color']))

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    # Time-resolved PCA: colour by epoch
    for ei, ep in enumerate(_epochs):
        mask = frame_epoch == ei
        if mask.any():
            axes[0].scatter(emb_t[mask, 0], emb_t[mask, 1],
                            color=ep['color'], s=8, alpha=0.7, linewidths=0,
                            label=ep['label'] if ei == 0 or _epochs[ei-1]['label'] != ep['label'] else '')
    if burst_t.any():
        axes[0].scatter(emb_t[burst_t, 0], emb_t[burst_t, 1],
                        c='crimson', s=20, alpha=0.9, linewidths=0, label='burst', zorder=3)
    axes[0].legend(fontsize=7, markerscale=2)
    axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
    axes[0].set_title('Time-resolved PCA (coloured by epoch)')
    sc2 = axes[1].scatter(emb_c[:, 0], emb_c[:, 1], c=_bp, cmap='YlOrRd',
                          s=50, edgecolors='k', linewidths=0.4)
    for i, rp in enumerate(rprops):
        axes[1].annotate(str(rp.label), (emb_c[i,0], emb_c[i,1]), fontsize=6,
                         ha='center', va='bottom', color='gray')
    plt.colorbar(sc2, ax=axes[1], label='Burst participation', shrink=0.8)
    axes[1].set_xlabel(f'PC1 ({ev_c[0]:.1f}%)'); axes[1].set_ylabel(f'PC2 ({ev_c[1]:.1f}%)')
    axes[1].set_title('Cell-resolved PCA')
    p1v, p99v = np.percentile(mean_img, [1, 99])
    axes[2].imshow(mean_img, cmap='gray', vmin=p1v, vmax=p99v)
    load_norm = plt.Normalize(vmin=-np.abs(emb_c[:,0]).max(), vmax=np.abs(emb_c[:,0]).max())
    load_cmap = cm.get_cmap('coolwarm')
    for i, rp in enumerate(rprops):
        y, x = rp.centroid
        axes[2].plot(x, y, 'o', color=load_cmap(load_norm(emb_c[i,0])), ms=7, mew=0.5, mec='k')
    sm = cm.ScalarMappable(cmap=load_cmap, norm=load_norm)
    plt.colorbar(sm, ax=axes[2], label='PC1 loading', shrink=0.8)
    axes[2].set_title('Spatial map -- PC1'); axes[2].axis('off')
    plt.suptitle(exp_dir.name, fontsize=9); plt.tight_layout()
    _save(fig, '05_pca.png')

    if RENDER_VIDEO:
        # ---- ROI activity video ---------------------------------------------
        def _probe_nvenc():
            try:
                r = subprocess.run(
                    ['ffmpeg','-f','lavfi','-i','nullsrc=s=64x64:r=1',
                     '-vframes','1','-c:v','h264_nvenc','-f','null','-'],
                    capture_output=True, timeout=10)
                return r.returncode == 0
            except Exception: return False
        _nvenc = _probe_nvenc()
        _codec = 'h264_nvenc' if _nvenc else 'libx264'
        _xargs = ['-preset','fast','-b:v','8M'] if _nvenc else ['-preset','fast','-crf','18']
    
        _p1, _p99 = np.percentile(mean_img, [1, 99])
        _bg3 = np.stack([np.clip((mean_img-_p1)/(_p99-_p1+1e-9),0,1).astype(np.float32)]*3, axis=-1)
        _cmap_v   = cm.get_cmap(ROI_VIDEO_CMAP)
        _ci       = (roi_mask-1).astype(np.int32)[roi_mask>0]
        _rys, _rxs = np.where(roi_mask > 0)
        _vmin_v   = float(np.percentile(dff,1))  if DFF_VMIN is None else float(DFF_VMIN)
        _vmax_v   = float(np.percentile(dff,99)) if DFF_VMAX is None else float(DFF_VMAX)
        _vmax_v   = _vmin_v + (_vmax_v - _vmin_v) * DFF_SCALE
        _dn       = np.clip((dff - _vmin_v) / (_vmax_v - _vmin_v + 1e-9), 0, 1)
        T_r       = T if MAX_VIDEO_FRAMES is None else min(T, MAX_VIDEO_FRAMES)
    
        def _rframe(t):
            fr = _bg3.copy()
            c  = _cmap_v(_dn[_ci, t])[:,:3].astype(np.float32)
            fr[_rys, _rxs] = ROI_VIDEO_ALPHA*c + (1-ROI_VIDEO_ALPHA)*fr[_rys,_rxs]
            return (np.clip(fr,0,1)*255).astype(np.uint8)
    
        fig_v = plt.figure(figsize=(9,13), facecolor='k')
        gs_v  = gridspec.GridSpec(3,1,figure=fig_v,height_ratios=[5,2,1],
                                   hspace=0.06,left=0.07,right=0.91,top=0.97,bottom=0.05)
        ax_im   = fig_v.add_subplot(gs_v[0])
        ax_rast = fig_v.add_subplot(gs_v[1])
        ax_tr   = fig_v.add_subplot(gs_v[2])
        _im_h   = ax_im.imshow(_rframe(0), aspect='equal', interpolation='nearest')
        ax_im.axis('off')
        _sm_v = cm.ScalarMappable(cmap=_cmap_v, norm=Normalize(vmin=_vmin_v, vmax=_vmax_v))
        plt.colorbar(_sm_v, ax=ax_im, label='dF/F', shrink=0.35, pad=0.01, fraction=0.025)
        _ttl  = ax_im.set_title(f't = {time_axis[0]:.1f} s', color='white', fontsize=9, pad=3)
        ax_rast.set_facecolor('k')
        ax_rast.imshow(active_binary[:,:T_r].astype(np.uint8), aspect='auto', cmap='hot',
                       vmin=0, vmax=1, interpolation='nearest',
                       extent=[time_axis[0],time_axis[T_r-1],n_cells+0.5,0.5])
        for b in bursts:
            ax_rast.axvspan(b['t_start'],b['t_end'],color='steelblue',alpha=0.2,lw=0)
        for ep in _epochs:
            _et0 = float(ep['start_s'])
            _et1 = min(float(ep['end_s']), float(time_axis[T_r-1]))
            if _et0 >= float(time_axis[T_r-1]): break
            ax_rast.axvspan(_et0, _et1, color=ep['color'], alpha=0.10, lw=0, zorder=0)
            ax_tr.axvspan(_et0, _et1, color=ep['color'], alpha=0.10, lw=0, zorder=0)
        _rc, = ax_rast.plot([time_axis[0]]*2,[0.5,n_cells+0.5],color='cyan',lw=1.2,zorder=5)
        ax_rast.set_xlim(time_axis[0],time_axis[T_r-1]); ax_rast.set_ylim(n_cells+0.5,0.5)
        ax_rast.set_ylabel('Cell #',color='white',fontsize=7)
        ax_rast.tick_params(colors='white',labelsize=7,labelbottom=False)
        ax_rast.set_yticks(np.arange(1,n_cells+1,max(1,n_cells//8)))
        for sp in ax_rast.spines.values(): sp.set_edgecolor('gray')
        ax_tr.set_facecolor('k')
        ax_tr.fill_between(time_axis[:T_r],pop_activity[:T_r],alpha=0.55,color='steelblue')
        for b in bursts:
            ax_tr.axvspan(b['t_start'],b['t_end'],color='salmon',alpha=0.4,lw=0)
        _pc, = ax_tr.plot([time_axis[0]]*2,[0,pop_activity[:T_r].max()*1.15 or 0.05],
                          color='crimson',lw=1.5,zorder=5)
        ax_tr.set_xlim(time_axis[0],time_axis[T_r-1])
        ax_tr.set_ylim(0,max(pop_activity[:T_r].max()*1.2,0.05))
        ax_tr.set_xlabel('Time (s)',color='white',fontsize=8)
        ax_tr.set_ylabel('Frac. active',color='white',fontsize=7)
        ax_tr.tick_params(colors='white',labelsize=7)
        for sp in ax_tr.spines.values(): sp.set_edgecolor('gray')
    
        mp4_path = exp_dir / 'roi_activity_video.mp4'
        writer   = _anim.FFMpegWriter(fps=ROI_VIDEO_FPS, codec=_codec, extra_args=_xargs)
        _log(f'Rendering {T_r} frames ...')
        with writer.saving(fig_v, str(mp4_path), dpi=100):
            for t in range(T_r):
                _im_h.set_data(_rframe(t))
                _ttl.set_text(f't = {time_axis[t]:.1f} s')
                _rc.set_xdata([time_axis[t],time_axis[t]])
                _pc.set_xdata([time_axis[t],time_axis[t]])
                writer.grab_frame()
                if t % 100 == 0: print(f'    {t}/{T_r}', end='\r')
        plt.close(fig_v)
        _log(f'Video saved ({mp4_path.stat().st_size/1e6:.1f} MB)')

    # ---- Return summary stats -------------------------------------------
    durs = [b['duration_s'] for b in bursts]
    ibi  = ([bursts[i+1]['t_start']-bursts[i]['t_end'] for i in range(len(bursts)-1)]
            if len(bursts) > 1 else [])
    return {
        'exp'          : exp_dir.name,
        'genotype'     : _get_genotype(exp_dir.name),
        'n_cells'      : n_cells,
        'mean_corr'    : round(mean_corr, 4),
        'n_bursts'     : len(bursts),
        'burst_freq_hz': round(len(bursts)/t_max, 4) if t_max > 0 else 0,
        'mean_dur_s'   : round(float(np.mean(durs)),  3) if durs else float('nan'),
        'mean_ibi_s'   : round(float(np.mean(ibi)),   3) if ibi  else float('nan'),
        'mean_recruit' : round(float(np.mean([b['n_recruits'] for b in bursts])), 1) if bursts else 0,
        'epoch_bursts' : epoch_burst_counts,
    }

print('analyse_experiment() defined')

In [ ]:
try: import pandas as pd; _HAS_PD = True
except ImportError: _HAS_PD = False

xl_meta   = parse_excel_metadata(METADATA_XLSX)
xl_lookup = {m['exp_dir']: m for m in xl_meta}

# Find experiments that have completed pipeline output
exp_entries = []
for m in xl_meta:
    pipeline_out = base_root / m['exp_dir'] / 'pipeline_output'
    if (pipeline_out / 'dff.npy').exists():
        exp_entries.append((base_root / m['exp_dir'], m))
    else:
        print(f'SKIP (no pipeline output): {m["exp_dir"]}')

print(f'Base: {base_root}')
print(f'Ready to analyse: {len(exp_entries)} / {len(xl_meta)} experiments')
for d, m in exp_entries:
    print(f'  [{m["sheet"]:18s}]  {d.name}')

results = []
for exp_dir, meta in exp_entries:
    print(f"\n{'='*60}\nSTART: {exp_dir.name}")
    try:
        r = analyse_experiment(exp_dir, xl_meta=meta)
    except Exception as e:
        print(f'  ERROR: {e}'); traceback.print_exc()
        r = {'exp': exp_dir.name, 'genotype': _get_genotype(exp_dir.name), 'error': str(e)}
    results.append(r)

print(f"\n{'='*60}\nBATCH COMPLETE")
if _HAS_PD:
    _cols = ['exp','genotype','n_cells','mean_corr','n_bursts',
             'burst_freq_hz','mean_dur_s','mean_ibi_s','mean_recruit']
    summary_df = pd.DataFrame([{k:r.get(k,'') for k in _cols} for r in results if 'error' not in r])
    print(summary_df.to_string(index=False))
else:
    for r in results: print(r)

## Summary plots
Cross-experiment comparison coloured by genotype (WT = steelblue, KO = coral).
Includes per-epoch burst count heatmap showing which condition drives activity.

In [ ]:
valid = [r for r in results if 'error' not in r]
if not valid:
    print('No valid results to plot')
else:
    _GT = {'WT': 'steelblue', 'KO': 'coral', 'unknown': 'gray'}
    exps  = [r['exp']      for r in valid]
    gts   = [r['genotype'] for r in valid]
    cols  = [_GT.get(g,'gray') for g in gts]
    xlabs = [re.sub(r'_Binning_?2_2_250ms','', e) for e in exps]
    x     = np.arange(len(valid))

    # ── Fig 1: Cell count + mean correlation ────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(x, [r['n_cells'] for r in valid], color=cols, edgecolor='white')
    axes[0].set_xticks(x); axes[0].set_xticklabels(xlabs, rotation=30, ha='right', fontsize=8)
    axes[0].set_ylabel('N cells'); axes[0].set_title('Cell count')
    for s in ['top','right']: axes[0].spines[s].set_visible(False)
    axes[1].bar(x, [r['mean_corr'] for r in valid], color=cols, edgecolor='white')
    axes[1].set_xticks(x); axes[1].set_xticklabels(xlabs, rotation=30, ha='right', fontsize=8)
    axes[1].set_ylabel('Mean Pearson r'); axes[1].set_title('Mean pairwise co-activity')
    for s in ['top','right']: axes[1].spines[s].set_visible(False)
    _leg = [plt.Rectangle((0,0),1,1,fc=c,label=g) for g,c in _GT.items() if g in gts]
    axes[1].legend(handles=_leg, fontsize=9)
    plt.suptitle('Summary -- cells & co-activity', fontsize=12); plt.tight_layout()
    _savefig(fig, base_root / 'summary_01_cells_coactivity.png'); plt.show()

    # ── Fig 2: Burst statistics ──────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, (vals, ylabel, title) in zip(axes, [
        ([r['burst_freq_hz'] for r in valid], 'Burst frequency (Hz)',  'Burst frequency'),
        ([r['mean_dur_s']    for r in valid], 'Mean duration (s)',     'Burst duration'),
        ([r['mean_ibi_s']    for r in valid], 'Mean IBI (s)',          'Inter-burst interval'),
    ]):
        ax.bar(x, [v if str(v)!='nan' else 0 for v in vals], color=cols, edgecolor='white')
        ax.set_xticks(x); ax.set_xticklabels(xlabs, rotation=30, ha='right', fontsize=8)
        ax.set_ylabel(ylabel); ax.set_title(title)
        for s in ['top','right']: ax.spines[s].set_visible(False)
    plt.suptitle('Summary -- burst statistics', fontsize=12); plt.tight_layout()
    _savefig(fig, base_root / 'summary_02_burst_stats.png'); plt.show()

    # ── Fig 3: Population activity overlay ──────────────────────────────
    ncols = min(3, len(valid))
    nrows = int(np.ceil(len(valid)/ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*5, nrows*2.5), sharey=False)
    axes_flat = np.array(axes).flat
    for r, ax in zip(valid, axes_flat):
        exp_d = base_root / r['exp']
        ta    = np.load(exp_d / 'time_axis.npy')
        dff_  = np.load(exp_d / 'dff.npy')
        rw    = max(3, int(RMS_WINDOW_S * len(ta) / ta[-1])) if ta[-1]>0 else 3
        crms  = np.sqrt(uniform_filter1d(dff_**2, size=rw, axis=1, mode='nearest'))
        ab    = dff_ > (SPIKE_RMS_FACTOR * crms)
        pa    = ab.mean(axis=0)
        pr    = np.sqrt(uniform_filter1d(pa**2, size=rw, mode='nearest'))
        _shade_epochs([ax], float(ta[-1]),
                       _assign_epoch_colors(_epochs_from_meta(xl_meta[0])) if xl_meta else [],
                       alpha=0.18, add_labels=True)
        ax.fill_between(ta, pa, alpha=0.6, color=_GT.get(r['genotype'],'gray'))
        ax.plot(ta, BURST_RMS_FACTOR*pr, color='crimson', lw=0.8, ls='--')
        ax.set_title(xlabs[valid.index(r)], fontsize=8)
        ax.set_ylim(0, max(pa.max()*1.1, 0.05))
        ax.set_xlabel('Time (s)', fontsize=7); ax.set_ylabel('Frac. active', fontsize=7)
        ax.tick_params(labelsize=7)
        for s in ['top','right']: ax.spines[s].set_visible(False)
    for ax in list(axes_flat)[len(valid):]: ax.set_visible(False)
    plt.suptitle('Population activity traces  (red dashed = burst threshold)', fontsize=11)
    plt.tight_layout()
    _savefig(fig, base_root / 'summary_03_population_traces.png'); plt.show()

    # ── Fig 4: Per-epoch burst count heatmap ────────────────────────────
    _ref_epochs = _assign_epoch_colors(_epochs_from_meta(xl_meta[0])) if xl_meta else []
    ep_labels_uniq = list(dict.fromkeys(ep['label'] for ep in _ref_epochs))
    hm = np.zeros((len(valid), len(ep_labels_uniq)))
    for i, r in enumerate(valid):
        for j, lbl in enumerate(ep_labels_uniq):
            hm[i, j] = r.get('epoch_bursts', {}).get(lbl, 0)
    fig, ax = plt.subplots(figsize=(max(6, len(ep_labels_uniq)*2.5), max(4, len(valid)*0.6+1.5)))
    im = ax.imshow(hm, aspect='auto', cmap='YlOrRd', vmin=0)
    plt.colorbar(im, ax=ax, label='Burst count')
    ax.set_xticks(range(len(ep_labels_uniq))); ax.set_xticklabels(ep_labels_uniq, fontsize=9)
    ax.set_yticks(range(len(valid)));          ax.set_yticklabels(xlabs, fontsize=8)
    ax.set_title('Bursts per epoch per experiment', fontsize=11)
    for i in range(len(valid)):
        for j in range(len(ep_labels_uniq)):
            ax.text(j, i, str(int(hm[i,j])), ha='center', va='center', fontsize=9,
                    color='white' if hm[i,j] > hm.max()*0.6 else 'black')
    plt.tight_layout()
    _savefig(fig, base_root / 'summary_04_epoch_bursts.png'); plt.show()

    # ── Fig 5: WT vs KO grouped comparison ──────────────────────────────
    gt_groups = {}
    for r in valid: gt_groups.setdefault(r['genotype'], []).append(r)
    metrics_grp = [
        ('n_cells',      'N cells',             'Cell count'),
        ('mean_corr',    'Mean Pearson r',       'Co-activity'),
        ('burst_freq_hz','Burst freq. (Hz)',     'Burst frequency'),
        ('mean_dur_s',   'Mean duration (s)',    'Burst duration'),
    ]
    fig, axes = plt.subplots(1, len(metrics_grp), figsize=(14, 5))
    rng = np.random.default_rng(42)
    for ax, (key, ylabel, title) in zip(axes, metrics_grp):
        for gi, (gt, rlist) in enumerate(sorted(gt_groups.items())):
            vals = [r[key] for r in rlist if str(r.get(key,'nan'))!='nan']
            xpos = gi + rng.uniform(-0.15, 0.15, len(vals))
            ax.scatter(xpos, vals, color=_GT.get(gt,'gray'), s=60,
                       edgecolors='k', linewidths=0.5, zorder=3)
            if vals:
                ax.plot([gi-0.2,gi+0.2],[np.mean(vals)]*2,color='k',lw=2,zorder=4)
        ax.set_xticks(range(len(gt_groups)))
        ax.set_xticklabels(sorted(gt_groups.keys()), fontsize=10)
        ax.set_ylabel(ylabel); ax.set_title(title)
        for s in ['top','right']: ax.spines[s].set_visible(False)
    plt.suptitle('WT vs KO  (dots = experiments, bar = mean)', fontsize=11); plt.tight_layout()
    _savefig(fig, base_root / 'summary_05_wt_vs_ko.png'); plt.show()

    print(f'\nSummary plots saved to {base_root}')